# 6. Evidence synthesis and footprint recommendation

This notebook visualises the trade-off between feature detection, directional coherence, sample support and tilt agreement. It does not automatically declare the fraction with the strongest tilt relationship to be correct.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "seacofs_tilt_tools.py").exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError("Run inside seacofs_eddy_tilt_analysis or one of its subfolders")
for path in (ANALYSIS_ROOT, HERE):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import seacofs_tilt_tools as tilt
import footprint_tools as ft
sns.set_theme(style="whitegrid", context="notebook")
palette = {"AE":"#c44e52", "CE":"#4c72b0"}

data = ft.load_cache()
filled = data[(data.footprint_kind == "Filled") & (data.pv_averaging == "nonlinear")].copy()
score = ft.footprint_scorecard(data)


In [ ]:
filled_score=score[score.footprint_kind.eq("Filled")].copy()
fig,axes=plt.subplots(2,2,figsize=(12,9),constrained_layout=True)
panels=[("median_p90","Upper-tail feature detection",True),
        ("median_coherence","Directional coherence",False),
        ("median_error_deg","Tilt-direction error",False),
        ("fraction_within_45","Tilt agreement within 45°",True)]
for ax,(column,title,higher) in zip(axes.flat,panels):
    for cyc,part in filled_score.groupby("Cyc"):
        ax.plot(part.ellipse_frac,part[column],marker="o",color=palette[cyc],label=cyc)
    ax.set(xlabel="Ellipse linear fraction",ylabel=column.replace("_"," "),title=title)
    if "p90" in column: ax.set_yscale("log")
axes[0,0].legend(frameon=False); fig.suptitle("Footprint decision dashboard"); plt.show()

In [ ]:
paired=ft.paired_with_reference(data)
ann=paired[paired.footprint_kind.eq("Annulus")].copy()
ann["exposure_gain_vs_full"] = ann.PV_grad_topo_mean_local_mag / ann.reference_PV_grad_topo_mean_local_mag
fig,ax=plt.subplots(figsize=(10,4.8),constrained_layout=True)
sns.boxplot(data=ann,x="footprint",y="exposure_gain_vs_full",hue="Cyc",palette=palette,showfliers=False,ax=ax)
ax.axhline(1,color="black",ls="--"); ax.tick_params(axis="x",rotation=30)
ax.set(xlabel="",ylabel="Annulus / full-ellipse exposure",title="Does a radial zone reveal concentrated forcing?"); plt.show()

## Decision rules

Prefer a footprint that has adequate grid-cell support, detects visually verified features, gives similar conclusions at adjacent fractions, and does not collapse net-vector coherence unnecessarily. Report the full ellipse’s net vector and non-cancelling exposure together. If outer annuli consistently detect verified topography missed by the inner core, retain annular exposure as a complementary diagnostic rather than replacing the net full-footprint vector.